# Tourism Recommendation System — Paper Replication

This notebook reconstructs the pipeline described in *“A Fusion of Recommendation Techniques to Deliver Personalized Tourism Experience”*.

**Important:** This is a rebuilt implementation, not the authors’ original code. Choices not specified in the paper are marked as assumptions in the code.

## How to use
1. Put this notebook in a folder (optionally together with `tourism_dataset_v2.csv`).
2. Make sure the required packages are installed: `numpy`, `pandas`, and `scikit-learn`.
3. Run the cells from top to bottom.
4. The notebook is **self-contained**: if `tourism_dataset_v2.csv` is not present, it is generated by the `generate_dataset()` function defined in the *Dataset generator* section below (same generator as in the project's main notebook, seed 42).
5. Result tables are saved to a `results/` folder next to the notebook.


## Imports and configuration

In [1]:
"""
Reconstruction of the pipeline in:
  Majeed et al., "A Fusion of Recommendation Techniques to Deliver Personalized
  Tourism Experience", IET Software 2026 (doi:10.1049/sfw2/9218059).

IMPORTANT: this is NOT the authors' code (they have not released it). It is rebuilt
from what the paper states, section by section:
  3.1.2  preprocessing            -> preprocess()
  3.1    Algorithm 1 (hybrid)     -> PAPER routing table + hybrid_matrix()
  3.2    CB / CF / KB modules     -> Modules
  3.4    DecisionTreeClassifier   -> decision_tree_section()
  3.1.6  Accuracy/Prec/Rec/F1 and the CTR/CR based TP/FP/TN/FN formulas
  4.1    per-category CTR / CR    -> ctr_cr_by_category()  (PROXY, see note there)
  4.3    synthetic vs real users  -> two evaluation scenarios ("ideal" vs "realistic")

Where the paper is silent (neighbourhood size, similarity weights, aggregation rule,
what a "click" is in a ratings dataset) a documented choice is made and marked ASSUMPTION.

Usage:  run this notebook top to bottom. DATA_PATH / OUTPUT_DIR are set in the last cell.
If DATA_PATH does not exist it is generated with generate_dataset() defined in this notebook.
"""
import argparse, os, warnings
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
SEED = 42
CATS = ["Adventure Spot", "Entertainment Venue", "Historical Landmark",
        "Museum", "Natural Wonder", "Urban Landmark"]

# Algorithm 1 exactly as printed in the paper (module sets per case).
PAPER = {"i": ["cb", "cf", "kb"], "ii": ["cb"], "iii": ["cf", "kb"], "iv": ["cf"]}
# Variant that only uses modules able to produce a number in each case.
ADAPT = {"i": ["cb", "cf", "kb"], "ii": ["kb"], "iii": ["kb"], "iv": ["cb", "kb"]}

## Dataset generator
Used only when `tourism_dataset_v2.csv` is not found. Identical to the generator in the project's main notebook, so this notebook no longer needs a separate `data_gen.py`.

In [2]:
import numpy as np, pandas as pd

def generate_dataset(seed=42, n_users=1791, n_ratings=9640, n_items=88):
    """Builds a fake-but-realistic tourism ratings table.
    A 'function' is a reusable block of code: you give it settings (seed, number of users...)
    and it returns a finished table (called a DataFrame)."""

    # The 'seed' makes the random numbers repeatable: same seed -> exactly the same dataset every run.
    rng = np.random.default_rng(seed)

    # The 6 attraction categories used in the paper
    CATS = ['Historical Landmark', 'Museum', 'Natural Wonder',
            'Adventure Spot', 'Urban Landmark', 'Entertainment Venue']

    # 20 real Pakistani cities with their true GPS position (latitude, longitude)
    # and a "type" that decides what kind of attractions are likely to be found there.
    CITIES = {
        'Lahore': (31.55, 74.34, 'heritage'), 'Multan': (30.16, 71.52, 'heritage'),
        'Taxila': (33.74, 72.79, 'heritage'), 'Peshawar': (34.01, 71.58, 'heritage'),
        'Bahawalpur': (29.40, 71.68, 'heritage'), 'Rawalpindi': (33.60, 73.04, 'heritage'),
        'Islamabad': (33.68, 73.05, 'urban'), 'Karachi': (24.86, 67.01, 'urban'),
        'Faisalabad': (31.42, 73.08, 'urban'), 'Sialkot': (32.49, 74.53, 'urban'),
        'Quetta': (30.18, 66.99, 'urban'),
        'Hunza': (36.32, 74.65, 'mountain'), 'Skardu': (35.30, 75.63, 'mountain'),
        'Gilgit': (35.92, 74.31, 'mountain'), 'Naran': (34.91, 73.65, 'mountain'),
        'Chitral': (35.85, 71.79, 'mountain'), 'Swat': (35.22, 72.43, 'mountain'),
        'Murree': (33.91, 73.39, 'mountain'), 'Nathia Gali': (34.07, 73.38, 'mountain'),
        'Gwadar': (25.12, 62.32, 'coast'),
    }

    # For each city type: the chance of each of the 6 categories (same order as CATS; each row adds up to 1).
    # Example: a 'mountain' city is 43% likely to get a Natural Wonder and 40% an Adventure Spot.
    cat_w = {
        'heritage': [.40, .30, .05, .05, .10, .10],
        'urban':    [.10, .15, .05, .05, .35, .30],
        'mountain': [.05, .02, .43, .40, .05, .05],
        'coast':    [.05, .05, .35, .25, .10, .20],
    }
    city_names = list(CITIES)

    # ---------------------------------------------------------------- 1) ATTRACTIONS (88)
    # Spread the 88 attractions as evenly as possible over the 20 cities (4 or 5 per city).
    per_city = np.full(len(city_names), n_items // len(city_names))
    per_city[: n_items - per_city.sum()] += 1
    rows = []
    for c, k in zip(city_names, per_city):            # loop over every city and how many attractions it gets
        lat, lon, typ = CITIES[c]
        for _ in range(k):                            # create k attractions in this city
            cat = rng.choice(CATS, p=cat_w[typ])      # pick a category using this city type's probabilities
            rows.append(dict(location_name=c, preferred_category=cat,
                             latitude=lat + rng.normal(0, .06),      # small random shift so attractions
                             longitude=lon + rng.normal(0, .06)))    # in one city are not at the same point
    items = pd.DataFrame(rows)
    items['attraction_id'] = [f'A{i+1:03d}' for i in range(n_items)]                    # A001 ... A088
    items['hotel_name'] = [f'{c} Grand Hotel {rng.integers(1, 4)}' for c in items.location_name]
    items['quality'] = rng.normal(0, .45, n_items)         # HIDDEN true quality: some places are just better
    items['popularity'] = rng.lognormal(0, .35, n_items)   # HIDDEN popularity: some places get more visitors

    # ---------------------------------------------------------------- 2) USERS (1791)
    users = pd.DataFrame({'user_id': [f'U{i+1:04d}' for i in range(n_users)]})     # U0001 ... U1791
    users['name'] = ['User ' + u[1:] for u in users.user_id]
    users['bias'] = rng.normal(0, .35, n_users)            # HIDDEN generosity: some users rate high, some low
    favs, stated = [], []
    for _ in range(n_users):
        # each user truly loves 1 category (55% of users) or 2 categories (45% of users)
        f = list(rng.choice(CATS, size=rng.choice([1, 2], p=[.55, .45]), replace=False))
        favs.append(f)
        # What the user TYPES on the registration form: their real favourites, but each answer is
        # replaced by a random category 15% of the time (people are not perfectly accurate).
        s = [x if rng.random() > .15 else rng.choice(CATS) for x in f]
        stated.append('|'.join(sorted(set(s))))            # store as text, e.g. "Museum|Natural Wonder"
    users['favs'] = favs
    users['stated_preferences'] = stated

    # ---------------------------------------------------------------- 3) HOW MANY RATINGS EACH USER GIVES
    # Every user gives at least 1 rating; on average about 5.4. We then nudge random users up or down
    # until the grand total is EXACTLY 9640 (the same as the paper).
    deg = 1 + rng.poisson((n_ratings - n_users) / n_users, n_users)
    deg = np.minimum(deg, n_items)
    while deg.sum() != n_ratings:
        j = rng.integers(0, n_users)
        if deg.sum() > n_ratings and deg[j] > 1: deg[j] -= 1
        elif deg.sum() < n_ratings and deg[j] < 14: deg[j] += 1

    # ---------------------------------------------------------------- 4) REVIEW TEXT
    # Sentence pieces. A review = [opening sentence matching the mood] + [sentence about the category] + [city].
    OPEN = {
        'neg': ["Not a great experience overall.", "I was quite disappointed.", "It did not live up to the hype.",
                "Would not recommend this place.", "Poor value for the time spent."],
        'neu': ["It was an average visit.", "A decent place, nothing extraordinary.", "Okay experience overall.",
                "Some good points and some bad ones.", "Fine for a short stop."],
        'pos': ["Absolutely loved this place!", "A wonderful and memorable visit.", "Highly recommended to every traveller.",
                "One of the best stops of my trip.", "Great experience, worth every minute."],
    }
    TOPIC = {
        'Historical Landmark': ["The old fort walls and heritage architecture tell a rich story.",
                                "Ancient ruins and centuries of history everywhere.",
                                "The guide explained the dynasty and the historic monuments well."],
        'Museum': ["The exhibits and gallery collections were well organised.",
                   "Artifacts, paintings and displays covered local culture nicely.",
                   "The curators arranged the museum halls thoughtfully."],
        'Natural Wonder': ["The valley, lake and mountain scenery were breathtaking.",
                           "Waterfalls and forests made for stunning nature views.",
                           "Clean air, rivers and peaceful natural beauty."],
        'Adventure Spot': ["The trekking trail and zip-line were thrilling.",
                           "Great hiking, camping and climbing activities.",
                           "Adrenaline all day with rafting and jeep safaris."],
        'Urban Landmark': ["The city square, towers and modern architecture are impressive.",
                           "Busy markets, streets and skyline views downtown.",
                           "A lively urban landmark with cafes all around."],
        'Entertainment Venue': ["The theme park rides and live shows were fun for the family.",
                                "Concerts, cinema and amusement games for all ages.",
                                "Great shows, food stalls and entertainment for kids."],
    }
    def review(rating, cat, city):
        s = rating
        # 25% of the time the mood of the text is one star away from the rating, so the review
        # is NOT a perfect copy of the rating (a perfect copy would be "data leakage" = cheating).
        if rng.random() < .25: s = int(np.clip(rating + rng.choice([-1, 1]), 1, 5))
        b = 'neg' if s <= 2 else 'neu' if s == 3 else 'pos'      # 1-2 stars = negative, 3 = neutral, 4-5 = positive
        return f"{rng.choice(OPEN[b])} {rng.choice(TOPIC[cat])} Visited {city}."

    # ---------------------------------------------------------------- 5) GENERATE EVERY RATING
    it_cat = items.preferred_category.values
    out = []
    for ui, u in users.iterrows():                      # go through users one by one
        # People visit what they like: attractions in the user's favourite categories are 3x more likely to be chosen.
        match_bonus = np.where(np.isin(it_cat, u.favs), 3.0, 1.0)
        p = items.popularity.values * match_bonus       # chance of visiting = popularity x bonus
        p = p / p.sum()                                 # turn into proper probabilities that add up to 1
        chosen = rng.choice(n_items, size=deg[ui], replace=False, p=p)   # which attractions this user rated
        for ii in chosen:
            it = items.iloc[ii]
            match = float(it.preferred_category in u.favs)   # 1 if this is one of the user's favourite categories
            # THE RATING RULE (the "hidden truth" the recommender must discover):
            #   base 2.9  + 0.8 if it is a favourite category  + 0.8 x attraction quality
            #   + the user's own generosity + a bit of random noise
            latent = 2.9 + .8 * match + .8 * it.quality + u.bias + rng.normal(0, .65)
            r = int(np.clip(np.round(latent), 1, 5))         # round to a whole number between 1 and 5 stars
            out.append(dict(user_id=u.user_id, name=u['name'], attraction_id=it.attraction_id,
                            hotel_name=it.hotel_name, rating=r, location_name=it.location_name,
                            preferred_category=it.preferred_category,
                            latitude=round(it.latitude, 6), longitude=round(it.longitude, 6),
                            review=review(r, it.preferred_category, it.location_name),
                            # random date between 1 Jan 2023 and 31 Dec 2025
                            timestamp=str(pd.Timestamp('2023-01-01') + pd.Timedelta(days=int(rng.integers(0, 1095)))).split()[0],
                            user_stated_preferences=u.stated_preferences))
    # shuffle the rows so they are not ordered by user, then return the finished table
    df = pd.DataFrame(out).sample(frac=1, random_state=seed).reset_index(drop=True)
    return df

## Load and preprocess the dataset

In [3]:
def load(path):
    if os.path.exists(path):
        return pd.read_csv(path)
    df = generate_dataset(seed=SEED)          # defined in the "Dataset generator" section above
    df.to_csv(path, index=False)
    return df


def preprocess(df):
    """Paper 3.1.2 (i)-(iii): cleaning, missing values, min-max normalisation."""
    n0 = len(df)
    df = df.drop_duplicates(subset=["user_id", "attraction_id", "timestamp"])
    df = df.dropna(subset=["user_id", "attraction_id", "rating"])        # list-wise deletion
    df = df[df.rating.between(1, 5)]
    df = df[df.review.fillna("").str.split().str.len() >= 3]             # <3-word reviews dropped
    lo, hi = df.rating.min(), df.rating.max()
    df["rating"] = 1 + 4 * (df.rating - lo) / (hi - lo)                  # min-max to the 1-5 scale
    df["rating"] = df.rating.round().astype(int)
    print(f"[preprocess] {n0} -> {len(df)} rows")
    return df.reset_index(drop=True)

## Geographic distance and context

In [4]:
def haversine_matrix(lat, lon):
    """Paper 3.1.2 (iv): pairwise haversine distance in km."""
    la, lo = np.radians(lat)[:, None], np.radians(lon)[:, None]
    dlat, dlon = la - la.T, lo - lo.T
    a = np.sin(dlat / 2) ** 2 + np.cos(la) * np.cos(la.T) * np.sin(dlon / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a))


class Ctx:
    """Index maps + static content that is known even for brand-new users/items."""
    def __init__(self, df):
        self.users = sorted(df.user_id.unique()); self.items = sorted(df.attraction_id.unique())
        uix = {u: k for k, u in enumerate(self.users)}; iix = {a: k for k, a in enumerate(self.items)}
        df["u"], df["i"] = df.user_id.map(uix), df.attraction_id.map(iix)
        self.nU, self.nI = len(self.users), len(self.items)
        info = df.groupby("i").agg(cat=("preferred_category", "first"), city=("location_name", "first"),
                                   lat=("latitude", "first"), lon=("longitude", "first")).reindex(range(self.nI))
        self.info = info
        self.cat_idx = np.array([CATS.index(c) for c in info.cat])
        self.dist = haversine_matrix(info.lat.values, info.lon.values)
        stated = np.zeros((self.nU, len(CATS)))                           # registration-form answers
        for u, s in df.groupby("u").user_stated_preferences.first().items():
            for c in s.split("|"):
                stated[u, CATS.index(c)] = 1
        self.match = stated[:, self.cat_idx]                              # users x items (1 = preferred category)

## Content-based, collaborative, and knowledge-based modules

In [5]:
# ----------------------------------------------------------------------------- modules
class Modules:
    """Trains CB, CF, KB on `train`; each attribute is a users x items matrix, NaN = cannot predict."""
    def __init__(self, ctx, train, k_nn=30, cb_lam=1.0):
        self.c, self.tr, self.k, self.lam = ctx, train, k_nn, cb_lam
        nU, nI = ctx.nU, ctx.nI
        R = np.full((nU, nI), np.nan); R[train.u, train.i] = train.rating
        self.M = ~np.isnan(R); self.gm = train.rating.mean()
        n_u, n_i = self.M.sum(1), self.M.sum(0)
        self.known_u, self.known_i = n_u > 0, n_i > 0
        self.um = np.where(self.known_u, np.nansum(R, 1) / np.maximum(n_u, 1), self.gm)
        self.im = (np.nansum(R, 0) + 5 * self.gm) / (n_i + 5)             # shrunk item mean (case base)
        self.Z = np.where(self.M, R - self.um[:, None], 0.0)              # user-mean-centred ratings
        self._kb(); self._cb(); self._cf()

    def _kb(self):                      # 3.2.3: case base + domain rule "preferred category"
        tr, c = self.tr, self.c
        m = c.match[tr.u.values, tr.i.values] == 1
        self.delta = tr.rating[m].mean() - tr.rating[~m].mean()
        eff = np.where(self.known_i, self.im - self.gm, 0.0)
        self.kb = self.gm + eff[None, :] + self.delta * c.match

    def _cb(self):                      # 3.2.5: item repository + item-similarity scoring
        c = self.c
        onehot = np.eye(len(CATS))[c.cat_idx]
        docs = self.tr.groupby("i").review.apply(" ".join).reindex(range(c.nI)).fillna("")
        tf = TfidfVectorizer(stop_words="english", min_df=3).fit_transform(docs).toarray()   # TF-IDF (3.1.2 v)
        tf = tf / (np.linalg.norm(tf, axis=1, keepdims=True) + 1e-9) * 0.7
        content = cosine_similarity(np.hstack([onehot, tf]))
        geo = 1.0 / (1.0 + c.dist / 100.0)                                # haversine -> similarity (ASSUMPTION)
        W = 0.7 * content + 0.3 * geo; np.fill_diagonal(W, 0); W = W ** 2  # ASSUMPTION: weights/power
        num, den = self.Z @ W.T, self.M.astype(float) @ W.T
        pred = self.um[:, None] + num / (den + self.lam)
        self.cb = np.where(self.known_u[:, None] & (den > 0), np.clip(pred, 1, 5), np.nan)

    def _cf(self):                      # 3.2.4: user-based k-NN on rating behaviour (3.1.2 iii names user-based k-NN)
        S = cosine_similarity(self.Z); np.fill_diagonal(S, 0); S[S < 0] = 0
        drop = np.argpartition(-S, self.k, axis=1)[:, self.k:]            # keep top-k neighbours
        np.put_along_axis(S, drop, 0, axis=1)
        num, den = S @ self.Z, S @ self.M.astype(float)
        with np.errstate(invalid="ignore", divide="ignore"):
            pred = self.um[:, None] + num / np.where(den > 0, den, np.nan)
        self.cf = np.where(self.known_u[:, None] & self.known_i[None, :], np.clip(pred, 1, 5), np.nan)

## Hybrid routing logic

In [6]:
def case_matrix(m):
    ku, ki = m.known_u[:, None], m.known_i[None, :]
    return np.where(ku & ki, "i", np.where(~ku & ki, "ii", np.where(~ku & ~ki, "iii", "iv")))


def hybrid_matrix(m, table):
    """Algorithm 1: per case average the prescribed modules that returned a value (aggregation = mean, ASSUMPTION)."""
    cases, out = case_matrix(m), np.full((m.c.nU, m.c.nI), np.nan)
    for c, names in table.items():
        with np.errstate(all="ignore"):
            avg = np.nanmean(np.stack([getattr(m, n) for n in names]), axis=0)
        out = np.where(cases == c, avg, out)
    fallback = np.isnan(out)
    out[fallback] = m.gm                                                   # nothing could predict -> global mean
    return out, cases, fallback

## Evaluation metrics

In [7]:
# ----------------------------------------------------------------------------- metrics
rmse = lambda y, p: float(np.sqrt(np.mean((y - p) ** 2)))
mae = lambda y, p: float(np.mean(np.abs(y - p)))


def cls_metrics(y, yhat):
    return dict(Accuracy=accuracy_score(y, yhat), Precision=precision_score(y, yhat, zero_division=0),
                Recall=recall_score(y, yhat, zero_division=0), F1=f1_score(y, yhat, zero_division=0))


def paper_confusion(ctr, cr, n):
    """The paper's 3.1.6 formulas. NOTE: these multiply RATES by the sample count; they are not
    counts of correctly/incorrectly classified items, which is why the paper's metrics are hard to interpret."""
    TP, FP = ctr * cr * n, (1 - ctr) * cr * n
    TN, FN = (1 - ctr) * (1 - cr) * n, ctr * (1 - cr) * n
    acc = (TP + TN) / (TP + FP + TN + FN); pre = TP / (TP + FP); rec = TP / (TP + FN)
    return dict(TP=TP, FP=FP, TN=TN, FN=FN, Accuracy=acc, Precision=pre, Recall=rec,
                F1=2 * pre * rec / (pre + rec))

## Train/test split

In [8]:
# ----------------------------------------------------------------------------- splits
def split(df, ctx, scenario, seed=SEED):
    r = np.random.default_rng(seed)
    if scenario == "realistic":          # new users + new items present in test (cold start)
        cu = set(r.choice(ctx.nU, int(.10 * ctx.nU), replace=False)); ci = set(r.choice(ctx.nI, 8, replace=False))
        forced = df.u.isin(cu) | df.i.isin(ci)
        rest = df[~forced]; t = r.random(len(rest)) < .2
        return rest[~t].copy(), pd.concat([df[forced], rest[t]]).copy()
    msk = r.random(len(df)) < .2          # "ideal": random 80/20 and only warm test pairs (no cold start)
    tr, te = df[~msk].copy(), df[msk].copy()
    te = te[te.u.isin(set(tr.u)) & te.i.isin(set(tr.i))]
    return tr, te

## Model evaluation

In [9]:
# ----------------------------------------------------------------------------- evaluation
def evaluate(ctx, tr, te, tag, thr=3.5):
    m = Modules(ctx, tr)
    ut, it, yt = te.u.values, te.i.values, te.rating.values
    cases = case_matrix(m)[ut, it]
    mats = {"Global mean": np.full((ctx.nU, ctx.nI), m.gm),
            "CB": np.where(np.isnan(m.cb), m.gm, m.cb), "CF": np.where(np.isnan(m.cf), m.gm, m.cf), "KB": m.kb}
    hp, _, fbp = hybrid_matrix(m, PAPER); ha, _, fba = hybrid_matrix(m, ADAPT)
    mats["Hybrid (paper Alg.1)"], mats["Hybrid (adaptive)"] = hp, ha
    rows = []
    for name, M in mats.items():
        p = M[ut, it]; row = dict(Scenario=tag, Method=name, RMSE=rmse(yt, p), MAE=mae(yt, p))
        row.update(cls_metrics((yt >= 4).astype(int), (p >= thr).astype(int)))        # relevant = rating>=4
        for c in ["i", "ii", "iii", "iv"]:
            s = cases == c; row[f"RMSE_case_{c}"] = rmse(yt[s], p[s]) if s.any() else np.nan
        rows.append(row)
    n_case = {c: int((cases == c).sum()) for c in ["i", "ii", "iii", "iv"]}
    print(f"[{tag}] train={len(tr)} test={len(te)} cases={n_case} | "
          f"Alg.1 had no usable module for {fbp[ut, it].mean():.1%} of test pairs (adaptive: {fba[ut, it].mean():.1%})")
    return m, mats, pd.DataFrame(rows)

## CTR and conversion-rate proxy

In [10]:
def ctr_cr_by_category(ctx, m, mats, tr, te, topn=5):
    """PROXY for paper 4.1 (their app logs are unavailable). For every test user recommend the top-N unseen
    items. impression = item shown; click = the user really rated it in the held-out data;
    conversion = click AND rating>=4.  CTR = clicks/impressions, CR = conversions/clicks (x100)."""
    seen = tr.groupby("u").i.apply(set).to_dict()
    truth = te.set_index(["u", "i"]).rating.to_dict()
    users = sorted(set(te.u))
    rows = []
    for name in ["CB", "CF", "KB", "Hybrid (paper Alg.1)", "Hybrid (adaptive)"]:
        imp = np.zeros(len(CATS)); clk = np.zeros(len(CATS)); cnv = np.zeros(len(CATS))
        for u in users:
            sc = mats[name][u] + 1e-6 * np.random.default_rng(u).random(ctx.nI)
            sc[list(seen.get(u, []))] = -np.inf
            for i in np.argsort(-sc)[:topn]:
                k = ctx.cat_idx[i]; imp[k] += 1
                if (u, i) in truth:
                    clk[k] += 1; cnv[k] += truth[(u, i)] >= 4
        for k, c in enumerate(CATS):
            rows.append(dict(Method=name, Category=c, Impressions=int(imp[k]), Clicks=int(clk[k]),
                             CTR=100 * clk[k] / imp[k] if imp[k] else np.nan,
                             CR=100 * cnv[k] / clk[k] if clk[k] else np.nan))
    return pd.DataFrame(rows)

## Decision tree evaluation

In [11]:
def decision_tree_section(df, ctx, depth=8):
    """Paper 3.4: DecisionTreeClassifier, 80/20 split, features = attraction attributes (location, category).
    The paper does not list the target encoding, so both the 5-class rating and the binary 'relevant' target are shown.
    A 'leaky' variant (user/item means computed INCLUDING the row's own rating) is added to show how
    inflated accuracy can appear when the label leaks into the features."""
    X = pd.DataFrame(np.eye(len(CATS))[ctx.cat_idx[df.i.values]], columns=CATS)
    X["lat"], X["lon"] = ctx.info.lat.values[df.i.values], ctx.info.lon.values[df.i.values]
    X["pref_match"] = ctx.match[df.u.values, df.i.values]
    leaky = X.copy()
    leaky["user_mean"] = df.groupby("u").rating.transform("mean").values       # includes own rating
    leaky["item_mean"] = df.groupby("i").rating.transform("mean").values
    r = np.random.default_rng(SEED); tr_mask = r.random(len(df)) < .8
    out = []
    for fname, F in [("attribute features (paper 3.4)", X), ("LEAKY features (diagnostic only)", leaky)]:
        for tname, y in [("rating 1-5", df.rating.values), ("relevant (>=4)", (df.rating.values >= 4).astype(int))]:
            clf = DecisionTreeClassifier(max_depth=depth, random_state=SEED).fit(F[tr_mask], y[tr_mask])
            p = clf.predict(F[~tr_mask]); avg = "weighted" if tname.startswith("rating") else "binary"
            out.append(dict(Features=fname, Target=tname, Accuracy=accuracy_score(y[~tr_mask], p),
                            Precision=precision_score(y[~tr_mask], p, average=avg, zero_division=0),
                            Recall=recall_score(y[~tr_mask], p, average=avg, zero_division=0),
                            F1=f1_score(y[~tr_mask], p, average=avg, zero_division=0)))
    return pd.DataFrame(out)

## Run the full experiment
This cell loads or generates the dataset, preprocesses it, evaluates both scenarios, and saves the result tables.

In [12]:
from pathlib import Path

DATA_PATH = "tourism_dataset_v2.csv"
OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(exist_ok=True)

df = preprocess(load(DATA_PATH))
ctx = Ctx(df)

print(f"[data] {ctx.nU} users x {ctx.nI} attractions, "
      f"sparsity {1 - len(df) / (ctx.nU * ctx.nI):.1%}")

results = []
for scenario in ["ideal", "realistic"]:
    train, test = split(df, ctx, scenario)
    model, matrices, metrics = evaluate(ctx, train, test, scenario)
    results.append(metrics)

    if scenario == "realistic":
        ctr_cr = ctr_cr_by_category(ctx, model, matrices, train, test)
        ctr_cr.to_csv(OUTPUT_DIR / "ctr_cr_by_category.csv", index=False)

results = pd.concat(results, ignore_index=True)
results.to_csv(OUTPUT_DIR / "metrics.csv", index=False)

print("\n=== Rating error and relevance classification ===")
display(results.round(3))

print("\n=== Proxy CTR/CR per category: Hybrid and KB ===")
display(ctr_cr[ctr_cr.Method.isin(["Hybrid (paper Alg.1)", "KB"])].round(1))

print("\n=== Decision tree evaluation ===")
decision_tree_results = decision_tree_section(df, ctx)
decision_tree_results.to_csv(OUTPUT_DIR / "decision_tree.csv", index=False)
display(decision_tree_results.round(3))

print("\n=== Paper's CTR/CR confusion formulas, applied to the paper's Table 2 hotel row ===")
display(pd.DataFrame([paper_confusion(0.873, 0.364, 1000)]).round(3))

print(f"\nSaved result files in: {OUTPUT_DIR.resolve()}")

[preprocess] 9640 -> 9640 rows
[data] 1791 users x 88 attractions, sparsity 93.9%


[ideal] train=7688 test=1931 cases={'i': 1931, 'ii': 0, 'iii': 0, 'iv': 0} | Alg.1 had no usable module for 0.0% of test pairs (adaptive: 0.0%)


[realistic] train=6226 test=3414 cases={'i': 1581, 'ii': 923, 'iii': 119, 'iv': 791} | Alg.1 had no usable module for 50.2% of test pairs (adaptive: 0.0%)



=== Rating error and relevance classification ===


,Scenario,Method,RMSE,MAE,Accuracy,Precision,Recall,F1,RMSE_case_i,RMSE_case_ii,RMSE_case_iii,RMSE_case_iv
0,ideal,Global mean,0.914,0.758,0.583,0.000,0.000,0.000,0.914,NaN,NaN,NaN
1,ideal,CB,0.960,0.767,0.618,0.546,0.503,0.524,0.960,NaN,NaN,NaN
2,ideal,CF,0.986,0.801,0.594,0.522,0.306,0.386,0.986,NaN,NaN,NaN
3,ideal,KB,0.839,0.672,0.648,0.559,0.740,0.637,0.839,NaN,NaN,NaN
4,ideal,Hybrid (paper Alg.1),0.827,0.664,0.676,0.613,0.606,0.610,0.827,NaN,NaN,NaN
5,ideal,Hybrid (adaptive),0.827,0.664,0.676,0.613,0.606,0.610,0.827,NaN,NaN,NaN
6,realistic,Global mean,0.943,0.771,0.556,0.000,0.000,0.000,0.933,0.968,1.052,0.914
7,realistic,CB,0.995,0.794,0.587,0.561,0.322,0.409,1.025,0.968,1.052,0.955
8,realistic,CF,0.987,0.802,0.552,0.483,0.114,0.185,1.027,0.968,1.052,0.914
9,realistic,KB,0.852,0.682,0.656,0.603,0.662,0.631,0.875,0.867,0.876,0.783



=== Proxy CTR/CR per category: Hybrid and KB ===


... (2 outputs left out)

,Features,Target,Accuracy,Precision,Recall,F1
0,attribute features (paper 3.4),rating 1-5,0.433,0.382,0.433,0.383
1,attribute features (paper 3.4),relevant (>=4),0.674,0.611,0.582,0.596
2,LEAKY features (diagnostic only),rating 1-5,0.496,0.489,0.496,0.478
3,LEAKY features (diagnostic only),relevant (>=4),0.742,0.696,0.665,0.680



=== Paper's CTR/CR confusion formulas, applied to the paper's Table 2 hotel row ===


,TP,FP,TN,FN,Accuracy,Precision,Recall,F1
0,317.772,46.228,80.772,555.228,0.399,0.873,0.364,0.514



Saved result files in: /home/claude/work/results
